# Training and Inference Time Estimator
This notebook estimates training time and inference time for any custom PyTorch architecture using only built-in torch tools. No external dependencies are required beyond PyTorch itself.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.utils.benchmark as benchmark
from torch.utils.flop_counter import FlopCounterMode
from torch.profiler import profile, ProfilerActivity, schedule
from typing import Optional

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
KNOWN_GPU_TFLOPS = {
    "rtx 2080 ti": 53.8,
    "rtx 2080 super": 43.2,
    "rtx 2080": 40.2,
}

SM_FP16_OPS = {
    (10, 0): 1024,
    (9, 0): 512,
    (8, 9): 256,
    (8, 6): 256,
    (8, 0): 512,
    (7, 5): 128,
    (7, 0): 128,
    (6, 1): 32,
    (6, 0): 32,
}


def detect_hardware(device_str: Optional[str] = None) -> dict:
    if device_str is None:
        device_str = "cuda" if torch.cuda.is_available() else "cpu"

    device = torch.device(device_str)
    info = {
        "device": device,
        "device_name": "",
        "peak_tflops": 0.0,
        "utilization_factor": 0.35,
        "num_gpus": 1,
        "vram_gb": 0.0,
        "tflops_source": "unknown",
    }

    if device.type == "cuda":
        props = torch.cuda.get_device_properties(device)
        info["device_name"] = props.name
        info["vram_gb"] = props.total_memory / (1024 ** 3)
        info["num_gpus"] = torch.cuda.device_count()
        info["compute_capability"] = f"{props.major}.{props.minor}"
        info["sm_count"] = props.multi_processor_count
        info["clock_ghz"] = props.clock_rate / 1e6
        info["memory_bandwidth_gbs"] = props.memory_clock_rate * props.memory_bus_width / 8e6

        name_lower = props.name.lower()
        if "rtx" in name_lower:
            info["utilization_factor"] = 0.30
        elif "a100" in name_lower or "h100" in name_lower:
            info["utilization_factor"] = 0.45

        # Step 1: lookup table with longest match for specificity
        matched_key = ""
        for key in KNOWN_GPU_TFLOPS:
            if key in name_lower and len(key) > len(matched_key):
                matched_key = key
        if matched_key:
            info["peak_tflops"] = KNOWN_GPU_TFLOPS[matched_key]
            info["tflops_source"] = "lookup_table"

        # Step 2: SM architecture estimate if not found in lookup
        if info["peak_tflops"] == 0.0:
            ops = SM_FP16_OPS.get((props.major, props.minor), 128)
            info["peak_tflops"] = round(
                (props.multi_processor_count * ops * info["clock_ghz"] * 2) / 1e3, 1
            )
            info["tflops_source"] = "sm_estimate"

        # Step 3: live microbenchmark as last resort
        if info["peak_tflops"] == 0.0:
            N = 4096
            a = torch.randn(N, N, dtype=torch.float16, device=device)
            b = torch.randn(N, N, dtype=torch.float16, device=device)
            for _ in range(3):
                torch.matmul(a, b)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(20):
                torch.matmul(a, b)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0
            info["peak_tflops"] = round(2 * N**3 * 20 / elapsed / 1e12, 1)
            info["tflops_source"] = "microbenchmark"

    elif device.type == "mps":
        info["device_name"] = "Apple MPS"
        info["peak_tflops"] = 11.0
        info["utilization_factor"] = 0.50
        info["tflops_source"] = "lookup_table"
    else:
        info["device_name"] = "CPU"
        info["peak_tflops"] = 1.0
        info["utilization_factor"] = 0.80
        info["tflops_source"] = "lookup_table"

    return info


hw = detect_hardware()

source_labels = {
    "lookup_table": "known GPU database",
    "sm_estimate": "SM architecture estimate",
    "microbenchmark": "live benchmark",
}

print("Hardware Summary")
print(f"Device: {hw['device_name']}")
print(f"Peak TFLOPs (FP16): {hw['peak_tflops']:.1f} [{source_labels.get(hw['tflops_source'], hw['tflops_source'])}]")
if hw["vram_gb"] > 0:
    print(f"VRAM: {hw['vram_gb']:.1f} GB")
    print(f"Memory bandwidth: {hw.get('memory_bandwidth_gbs', 0):.0f} GB/s")
    print(f"Compute capability: {hw.get('compute_capability', 'N/A')}")
    print(f"SM count: {hw.get('sm_count', 'N/A')}")
print(f"GPU count: {hw['num_gpus']}")
print(f"Utilization estimate: {hw['utilization_factor'] * 100:.0f}%")

## Define Your Architecture
Replace the `CustomTransformer` class below with your own model. The estimator works with any `nn.Module`.

In [ ]:
class CustomTransformer(nn.Module):
    """
    Replace this with your actual model architecture.
    The only requirement is a standard nn.Module with a
    forward() method that accepts a tensor input.
    """
    def __init__(
        self,
        vocab_size: int = 32000,
        hidden_dim: int = 768,
        num_layers: int = 12,
        num_heads: int = 12,
        ffn_dim: int = 3072,
        dropout: float = 0.1
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embedding = nn.Embedding(2048, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.output_proj = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.encoder(x)
        x = self.layer_norm(x)
        return self.output_proj(x)


device = hw["device"]

model = CustomTransformer(
    vocab_size=32000,
    hidden_dim=768,
    num_layers=12,
    num_heads=12,
    ffn_dim=3072
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size FP32: {total_params * 4 / 1024**2:.1f} MB")
print(f"Model size FP16: {total_params * 2 / 1024**2:.1f} MB")

## Dataset and Training Configuration

In [ ]:
NUM_SAMPLES = 100_000
BATCH_SIZE = 32
NUM_EPOCHS = 3
SEQUENCE_LENGTH = 512

steps_per_epoch = NUM_SAMPLES // BATCH_SIZE
total_steps = steps_per_epoch * NUM_EPOCHS

print(f"Samples per epoch: {NUM_SAMPLES:,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Steps per epoch: {steps_per_epoch:,}")
print(f"Total epochs: {NUM_EPOCHS}")
print(f"Total steps: {total_steps:,}")

dummy_input = torch.randint(0, 1000, (BATCH_SIZE, SEQUENCE_LENGTH), device=device)
dummy_target = torch.randint(0, 32000, (BATCH_SIZE, SEQUENCE_LENGTH), device=device)

## FLOPs Counting with torch.utils.flop_counter

In [ ]:
model.eval()

with FlopCounterMode(model, display=True) as flop_counter:
    with torch.no_grad():
        _ = model(dummy_input)

forward_flops = flop_counter.get_total_flops()

# Backward pass is approximately 2x the forward pass
# so total training FLOPs per step = forward * 3 * batch_size
training_flops_per_step = forward_flops * 3 * BATCH_SIZE
inference_flops_per_sample = forward_flops

print(f"Forward FLOPs per sample: {forward_flops:,}")
print(f"Training FLOPs per batch: {training_flops_per_step:,}")
print(f"Training FLOPs full run: {training_flops_per_step * total_steps:.3e}")

model.train()

## Theoretical Training Time Estimate

In [ ]:
def format_time(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f} seconds"
    elif seconds < 3600:
        return f"{seconds / 60:.1f} minutes"
    elif seconds < 86400:
        return f"{seconds / 3600:.2f} hours"
    else:
        return f"{seconds / 86400:.2f} days"


total_training_flops = training_flops_per_step * total_steps
effective_flops_per_sec = hw["peak_tflops"] * hw["utilization_factor"] * hw["num_gpus"] * 1e12
theoretical_training_seconds = total_training_flops / effective_flops_per_sec

print("Theoretical Training Time Estimate")
print(f"Effective throughput: {effective_flops_per_sec / 1e12:.1f} TFLOPs/s")
print(f"Total training FLOPs: {total_training_flops:.3e}")
print(f"Estimated time: {format_time(theoretical_training_seconds)}")

## Measured Training Step Time with torch.utils.benchmark

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()


def training_step():
    optimizer.zero_grad()
    output = model(dummy_input)
    loss = criterion(
        output.view(-1, output.size(-1)),
        dummy_target.view(-1)
    )
    loss.backward()
    optimizer.step()


timer = benchmark.Timer(
    stmt="training_step()",
    globals={"training_step": training_step},
    num_threads=1,
    label="Training step",
    sub_label=f"batch={BATCH_SIZE} seq={SEQUENCE_LENGTH}"
)

result = timer.timeit(number=20)
print(result)

ms_per_step = result.median * 1000
seconds_per_step = result.median
measured_training_seconds = seconds_per_step * total_steps

print(f"Measured time per step: {ms_per_step:.2f} ms")
print(f"Steps per epoch: {steps_per_epoch:,}")
print(f"Total steps: {total_steps:,}")
print(f"Estimated training time: {format_time(measured_training_seconds)}")

## Measured Inference Time with torch.utils.benchmark

In [ ]:
model.eval()

inference_timer_batch = benchmark.Timer(
    stmt="model(dummy_input)",
    globals={"model": model, "dummy_input": dummy_input},
    num_threads=1,
    label="Batch inference",
    sub_label=f"batch={BATCH_SIZE} seq={SEQUENCE_LENGTH}"
)

single_input = torch.randint(0, 1000, (1, SEQUENCE_LENGTH), device=device)
inference_timer_single = benchmark.Timer(
    stmt="model(single_input)",
    globals={"model": model, "single_input": single_input},
    num_threads=1,
    label="Single sample inference",
    sub_label=f"batch=1 seq={SEQUENCE_LENGTH}"
)

with torch.no_grad():
    result_batch = inference_timer_batch.timeit(number=50)
    result_single = inference_timer_single.timeit(number=50)

ms_batch = result_batch.median * 1000
ms_single = result_single.median * 1000

print(result_batch)
print(result_single)

print(f"Batch inference ({BATCH_SIZE} samples): {ms_batch:.2f} ms")
print(f"Single sample inference: {ms_single:.2f} ms")
print(f"Per-sample latency from batch: {ms_batch / BATCH_SIZE:.3f} ms")
print(f"Throughput: {BATCH_SIZE / result_batch.median:.0f} samples/sec")

model.train()

## Per-operator Profiling During Inference

In [ ]:
model.eval()

activities = [ProfilerActivity.CPU]
if device.type == "cuda":
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    schedule=schedule(wait=1, warmup=2, active=3),
    record_shapes=True,
    with_flops=True,
    profile_memory=True
) as prof:
    for step in range(6):
        with torch.no_grad():
            _ = model(dummy_input)
        prof.step()

sort_key = "cuda_time_total" if device.type == "cuda" else "cpu_time_total"
print("Top operators by CUDA time during inference")
print(prof.key_averages().table(sort_by=sort_key, row_limit=15))

model.train()

## Per-operator Profiling During Training

In [ ]:
activities = [ProfilerActivity.CPU]
if device.type == "cuda":
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    schedule=schedule(wait=1, warmup=2, active=3),
    record_shapes=True,
    with_flops=True,
    profile_memory=True
) as prof:
    for step in range(6):
        optimizer.zero_grad()
        output = model(dummy_input)
        loss = criterion(output.view(-1, output.size(-1)), dummy_target.view(-1))
        loss.backward()
        optimizer.step()
        prof.step()

sort_key = "cuda_time_total" if device.type == "cuda" else "cpu_time_total"
print("Top operators by CUDA time during training")
print(prof.key_averages().table(sort_by=sort_key, row_limit=15))

## Memory Usage

In [ ]:
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)

    optimizer.zero_grad()
    output = model(dummy_input)
    loss = criterion(output.view(-1, output.size(-1)), dummy_target.view(-1))
    loss.backward()
    optimizer.step()

    allocated = torch.cuda.memory_allocated(device) / 1024**3
    reserved = torch.cuda.memory_reserved(device) / 1024**3
    peak = torch.cuda.max_memory_allocated(device) / 1024**3
    vram = hw["vram_gb"]

    print(f"Memory allocated: {allocated:.2f} GB")
    print(f"Memory reserved: {reserved:.2f} GB")
    print(f"Peak allocated: {peak:.2f} GB")
    print(f"Total VRAM: {vram:.1f} GB")
    print(f"VRAM utilization: {peak / vram * 100:.1f}%")

    headroom = vram - peak
    if headroom < 1.0:
        print(f"Warning: only {headroom:.1f} GB headroom remaining, OOM risk at larger batches")
    else:
        print(f"Headroom remaining: {headroom:.1f} GB")
else:
    print("Memory stats are only available for CUDA devices")

## Full Summary Report

In [ ]:
ratio = measured_training_seconds / max(theoretical_training_seconds, 1e-9)

print("=" * 60)
print("TRAINING AND INFERENCE TIME ESTIMATION REPORT")
print("=" * 60)

print("\nHardware")
print(f"Device: {hw['device_name']}")
print(f"Peak TFLOPs (FP16): {hw['peak_tflops']:.1f}")
if hw["vram_gb"] > 0:
    print(f"VRAM: {hw['vram_gb']:.1f} GB")

print("\nModel")
print(f"Trainable params: {trainable_params:,}")
print(f"Forward FLOPs: {forward_flops:.3e}")

print("\nDataset")
print(f"Samples: {NUM_SAMPLES:,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Total steps: {total_steps:,}")

print("\nTraining Time")
print(f"Theoretical: {format_time(theoretical_training_seconds)}")
print(f"Measured: {format_time(measured_training_seconds)}")
print(f"Per step: {ms_per_step:.2f} ms")
print(f"Benchmark vs theory ratio: {ratio:.2f}x")

if ratio > 2.0:
    print("Note: actual time is significantly higher than theory.")
    print("This is likely due to memory bandwidth or data loading overhead.")
elif ratio < 0.5:
    print("Note: actual time is lower than theory.")
    print("The TFLOPs estimate may be conservative for this GPU.")

print("\nInference Time")
print(f"Batch ({BATCH_SIZE} samples): {ms_batch:.2f} ms")
print(f"Single sample: {ms_single:.2f} ms")
print(f"Per-sample latency: {ms_batch / BATCH_SIZE:.3f} ms")
print(f"Throughput: {BATCH_SIZE / result_batch.median:.0f} samples/sec")

if device.type == "cuda":
    print("\nMemory")
    print(f"Peak training VRAM: {peak:.2f} GB of {vram:.1f} GB ({peak / vram * 100:.1f}%)")

print("\n" + "=" * 60)